# Exploring a brewing yeast genome — `FLO11` variant graph

This notebook analyses the *FLO11* locus using a 9.1 kb fragment of
*S. cerevisiae* chrIX (positions 389,572–398,675).

| Gene | Function | Brewing relevance |
|------|----------|-------------------|
| **FLO11** (*MUC1/YIR019C*) | GPI-anchored cell-surface flocculin | Pseudohyphal growth, flocculation, biofilm |

**FLO11** is transcriptionally controlled by two converging MAPK pathways
via a composite *Filamentation Response Element* (FRE) in its promoter:

| Element | TF | Canonical sequence |
|---------|----|-----------------|
| **PRE** (Pheromone Response Element) | Ste12 | `TGAAACA` |
| **TCS** (Tec1 Consensus Sequence) | Tec1 | `CATTCC` / `CATTCT` |
| **FRE** | Ste12 + Tec1 heterodimer | PRE + TCS within ~100 bp |

**BRQ** is a brewing yeast strain.  We load the S288c reference sequence
and apply BRQ variant calls to build a variant graph encoding both haplotypes.

**Workflow:**
1. Import the reference FASTA and BRQ VCF
2. Plot the BRQ variant graph with gene annotations
3. Search for Ste12 PRE sites (both strands)
4. Search for Tec1 TCS sites (both variants, both strands)
5. Highlight all motifs in distinct colours
6. Detect FRE-like pairs — PRE + TCS within 100 bp

## Setup

The fixture files (`flo11_reference.fa`, `flo11_variants.vcf`, `flo11_reference.gff3`)
live in the same directory as this notebook.  A fresh temporary repository is
created each run so repeated executions don't accumulate state.

In [1]:
import pathlib
import re
import tempfile

import gen

EXAMPLES_DIR = pathlib.Path(".").resolve()
FASTA = EXAMPLES_DIR / "flo11_reference.fa"
VCF   = EXAMPLES_DIR / "flo11_variants.vcf"
GFF3  = EXAMPLES_DIR / "flo11_reference.gff3"

assert FASTA.exists(), f"Missing: {FASTA}"
assert VCF.exists(),   f"Missing: {VCF}"
assert GFF3.exists(),  f"Missing: {GFF3}"

WORK_DIR = pathlib.Path(tempfile.mkdtemp(prefix="gen-flo11-"))
print(f"Working in {WORK_DIR}")

repo = gen.Repository(str(WORK_DIR))
repo.import_fasta(str(FASTA), sample='reference')
repo.update_with_vcf(str(VCF), parent_samples=['reference'])  # creates BRQ sample

bgs = repo.get_block_groups()
print(f"{len(bgs)} block group(s) imported")
for b in bgs:
    print(f"  {b.sample_name:15s}  {b.name}")

bg_brq = next(b for b in bgs if b.sample_name == 'BRQ')
bg_ref = next(b for b in bgs if b.sample_name == 'reference')

Working in /var/folders/f8/8zf8xczs0pxf9_vfnlmqlx9h0000gn/T/gen-flo11-39w3_rv0
2 block group(s) imported
  reference        pad1_fdc1_region
  BRQ              pad1_fdc1_region


## Plot the BRQ variant graph

The graph encodes both the S288c reference path and the BRQ-specific variant
path.  Nodes shared between haplotypes appear once; divergent nodes form bubbles.

`fig.show_path()` draws the BRQ path as a coloured ribbon over the graph
topology.  FLO11 is on the minus strand — its coding sequence occupies the
left portion of the graph (positions 1–4104), with the promoter extending
rightward toward position ~9100.

In [2]:
fig_path = bg_brq.plot()
fig_path.show_path()

## Gene annotations from GFF3

`add_annotation_track_file()` projects GFF3 features onto the graph as
coloured bars below the canvas.  Filtering to `gene` rows shows FLO11 and
the flanking genes without redundant CDS / chromosome entries.

FLO11 spans positions 1–4104 (minus strand).  The region to the right
of FLO11 in the graph — positions ~4200 onward — is the large intergenic
promoter where regulatory elements are expected.

In [3]:
fig_genes = bg_brq.plot(rows=24)

fig_genes.add_annotation_track_file(
    file=str(GFF3),
    filter=lambda row: row.split("\t")[2] == "gene",
    name="genes",
)

print("Track panels:", fig_genes.annotation_tracks())

Track panels: ['genes']


## Ste12 PRE sites

The **Pheromone Response Element** (`TGAAACA`) is the canonical Ste12 binding
site.  Ste12 drives FLO11 expression through the MAPK (filamentous growth)
pathway.  It can also act as a monomer; PRE sites outside of FREs have been
reported in the FLO11 promoter.

Search is case-insensitive.  We search both strands explicitly:
- Forward: `TGAAACA`
- Reverse complement: `TGTTTCA`

Each hit is placed as an **inline annotation** on the canvas so its label
(`PRE-fwd:N` / `PRE-rc:N`) appears directly below the highlighted span.

In [13]:
pre_fwd[-1].start()

GraphPos(b087efb4[13472..13557] +81)

In [11]:
pre_fwd = bg_brq.search("tgaaaca")
pre_rev = bg_brq.search("tgtttca")
pre_hits = pre_fwd + pre_rev

print(f"PRE hits  forward (TGAAACA): {len(pre_fwd)}")
print(f"PRE hits  reverse (TGTTTCA): {len(pre_rev)}")
print(f"PRE hits  total:             {len(pre_hits)}")

fig_pre = bg_brq.plot(rows=28)
fig_pre.add_annotation_track_file(
    file=str(GFF3),
    filter=lambda row: row.split("\t")[2] == "gene",
    name="genes",
)
for i, h in enumerate(pre_fwd):
    fig_pre.add_annotation(gen.Annotation(h, name=f"PRE-fwd:{i}"))
for i, h in enumerate(pre_rev):
    fig_pre.add_annotation(gen.Annotation(h, name=f"PRE-rc:{i}"))

fig_pre.go_to(pre_fwd[-1].start())
print("Inline annotations:", fig_pre.inline_annotations())

PRE hits  forward (TGAAACA): 3
PRE hits  reverse (TGTTTCA): 4
PRE hits  total:             7


Inline annotations: ['PRE-fwd:0', 'PRE-fwd:1', 'PRE-fwd:2', 'PRE-rc:0', 'PRE-rc:1', 'PRE-rc:2', 'PRE-rc:3']


## Tec1 TCS sites

The **Tec1 Consensus Sequence** is the binding site for Tec1, which
heterodimerises with Ste12 to form the FRE complex.  Two variants are
reported in the literature:

| Variant | Forward | Reverse complement |
|---------|---------|-------------------|
| TCS-1 | `CATTCC` | `GGAATG` |
| TCS-2 | `CATTCT` | `AGAATG` |

Both variants are searched on both strands.  Inline annotations label each
hit by variant and strand (`TCS1-fwd`, `TCS1-rc`, `TCS2-fwd`, `TCS2-rc`)
so they can be distinguished on the canvas.

In [5]:
tcs1_fwd = bg_brq.search("cattcc")
tcs1_rev = bg_brq.search("ggaatg")
tcs2_fwd = bg_brq.search("cattct")
tcs2_rev = bg_brq.search("agaatg")
tcs_hits = tcs1_fwd + tcs1_rev + tcs2_fwd + tcs2_rev

print(f"TCS-1 hits  forward (CATTCC): {len(tcs1_fwd)}")
print(f"TCS-1 hits  reverse (GGAATG): {len(tcs1_rev)}")
print(f"TCS-2 hits  forward (CATTCT): {len(tcs2_fwd)}")
print(f"TCS-2 hits  reverse (AGAATG): {len(tcs2_rev)}")
print(f"TCS   total:                  {len(tcs_hits)}")

fig_tcs = bg_brq.plot(rows=28)
fig_tcs.add_annotation_track_file(
    file=str(GFF3),
    filter=lambda row: row.split("\t")[2] == "gene",
    name="genes",
)
for label, group in [
    ("TCS1-fwd", tcs1_fwd),
    ("TCS1-rc",  tcs1_rev),
    ("TCS2-fwd", tcs2_fwd),
    ("TCS2-rc",  tcs2_rev),
]:
    for i, h in enumerate(group):
        fig_tcs.add_annotation(gen.Annotation(h, name=f"{label}:{i}"))

print("Inline annotations:", fig_tcs.inline_annotations())

TCS-1 hits  forward (CATTCC): 0
TCS-1 hits  reverse (GGAATG): 3
TCS-2 hits  forward (CATTCT): 8
TCS-2 hits  reverse (AGAATG): 7
TCS   total:                  18


Inline annotations: ['TCS1-rc:0', 'TCS1-rc:1', 'TCS1-rc:2', 'TCS2-fwd:0', 'TCS2-fwd:1', 'TCS2-fwd:2', 'TCS2-fwd:3', 'TCS2-fwd:4', 'TCS2-fwd:5', 'TCS2-fwd:6', 'TCS2-fwd:7', 'TCS2-rc:0', 'TCS2-rc:1', 'TCS2-rc:2', 'TCS2-rc:3', 'TCS2-rc:4', 'TCS2-rc:5', 'TCS2-rc:6']


## PRE + TCS together

With both motif classes annotated, spatial proximity becomes the key question:
which PRE and TCS sites are close enough to form an FRE?

We use **annotation tracks** here — one track per motif class — so each hit
appears as a labelled bar in its own row below the graph.  This keeps the
canvas uncluttered while making gap distances easy to judge visually.

In [6]:
fig_combined = bg_brq.plot(rows=32)
fig_combined.add_annotation_track_file(
    file=str(GFF3),
    filter=lambda row: row.split("\t")[2] == "gene",
    name="genes",
)
fig_combined.add_annotation_track(
    [gen.Annotation(h, name=f"PRE:{i}") for i, h in enumerate(pre_hits)],
    name="PRE (Ste12)",
)
fig_combined.add_annotation_track(
    [gen.Annotation(h, name=f"TCS:{i}") for i, h in enumerate(tcs_hits)],
    name="TCS (Tec1)",
)

print("Tracks:", fig_combined.annotation_tracks())
print(f"PRE sites: {len(pre_hits)}   TCS sites: {len(tcs_hits)}")

Tracks: ['genes', 'PRE (Ste12)', 'TCS (Tec1)']
PRE sites: 7   TCS sites: 18


## FRE-like pairs — PRE + TCS within 100 bp

A **Filamentation Response Element** is defined by a PRE and TCS in close
proximity (~100 bp).  The Ste12–Tec1 heterodimer bridges both sites.

We approximate each hit's genomic coordinate from the `GraphLocus`
representation (`node[start..end]+offset`) and pair hits within 100 bp.

The FRE figure layers two annotation styles:
- **Track panels** (below canvas) — all PRE and TCS hits for context
- **Inline annotations** on the canvas — only the FRE pair members,
  labelled `FRE1-PRE` / `FRE1-TCS` etc. so paired sites are immediately
  identifiable by number

In [7]:
_LOCUS_RE = re.compile(r'\[(\d+)\.\.\d+\]\+(\d+)')

def _approx_start(locus):
    """node_start + intra-node offset from the first block of a GraphLocus."""
    m = _LOCUS_RE.search(repr(locus))
    return int(m.group(1)) + int(m.group(2)) if m else None

FRE_WINDOW = 100

fre_pairs = []
for pre in pre_hits:
    p = _approx_start(pre)
    if p is None:
        continue
    for tcs in tcs_hits:
        t = _approx_start(tcs)
        if t is None:
            continue
        if abs(p - t) <= FRE_WINDOW:
            fre_pairs.append((pre, tcs, abs(p - t)))

print(f"FRE-like pairs (PRE + TCS within {FRE_WINDOW} bp): {len(fre_pairs)}")
for n, (pre, tcs, dist) in enumerate(fre_pairs, 1):
    print(f"  FRE{n}  dist={dist:3d}  PRE {pre}")
    print(f"          TCS {tcs}")

fig_fre = bg_brq.plot(rows=36)
fig_fre.add_annotation_track_file(
    file=str(GFF3),
    filter=lambda row: row.split("\t")[2] == "CDS",
    name="genes",
)
# Track panels provide full PRE/TCS context below the canvas
fig_fre.add_annotation_track(
    [gen.Annotation(h, name=f"PRE:{i}") for i, h in enumerate(pre_hits)],
    name="PRE (Ste12)",
)
fig_fre.add_annotation_track(
    [gen.Annotation(h, name=f"TCS:{i}") for i, h in enumerate(tcs_hits)],
    name="TCS (Tec1)",
)
# Inline annotations on the canvas — FRE pair members only, numbered by pair
for n, (pre, tcs, _) in enumerate(fre_pairs, 1):
    fig_fre.add_annotation(gen.Annotation(pre, name=f"FRE{n}-PRE"))
    fig_fre.add_annotation(gen.Annotation(tcs, name=f"FRE{n}-TCS"))

print("\nInline annotations:", fig_fre.inline_annotations())
print("Track panels:      ", fig_fre.annotation_tracks())

FRE-like pairs (PRE + TCS within 100 bp): 3
  FRE1  dist= 48  PRE GraphLocus(b087efb4[41..54]+11 → b087efb4[55..93]+4, 3 blocks)
          TCS GraphLocus(72799c1a[0..13]+4 → 72799c1a[0..13]+10, 1 blocks)
  FRE2  dist=  4  PRE GraphLocus(88d631fe[0..1]+0 → b087efb4[13593..13912]+6, 2 blocks)
          TCS GraphLocus(72799c1a[0..13]+4 → 72799c1a[0..13]+10, 1 blocks)
  FRE3  dist=  4  PRE GraphLocus(00209b06[0..1]+0 → b087efb4[10318..10330]+6, 2 blocks)
          TCS GraphLocus(72799c1a[0..13]+4 → 72799c1a[0..13]+10, 1 blocks)



Inline annotations: ['FRE1-PRE', 'FRE1-TCS', 'FRE2-PRE', 'FRE2-TCS', 'FRE3-PRE', 'FRE3-TCS']
Track panels:       ['genes', 'PRE (Ste12)', 'TCS (Tec1)']


In [8]:
for a in fig_fre.inline_annotations():
    #fig_fre.go_to(a)
    print(a)

FRE1-PRE
FRE1-TCS
FRE2-PRE
FRE2-TCS
FRE3-PRE
FRE3-TCS


## Freeze for distribution

`fig.freeze()` captures the current widget state as a static PNG baked
into the `.ipynb` file.  Frozen widgets render on GitHub, nbviewer, and
other static viewers even without the `gen` module installed.

Run all cells above first, navigate each live widget to the desired view,
then run this cell to freeze them all.

In [9]:
for fig in [fig_path, fig_genes, fig_pre, fig_tcs, fig_combined, fig_fre]:
    break
    fig.freeze()

## Tips

**Promoter orientation** — FLO11 is on the minus strand.  PRE and TCS sites
on the minus strand (the RC queries) are the ones active in driving FLO11;
plus-strand hits in the same region would regulate a gene in the opposite
direction.

**FRE pair distance** — the `_approx_start` helper uses the node's reference
coordinate plus the intra-node offset.  In graph regions with variant bubbles
the coordinate may differ slightly from the linear reference position; treat
the distance as approximate.

**Building an index** — `repo.build_index(k=8)` speeds up repeated searches
over the same block groups.  Call it once after import; the index is picked
up automatically by all subsequent `bg.search()` calls.

**Extending the analysis** — the FLO11 promoter is unusually large (~3 kb)
and contains multiple FREs with distinct regulatory logic.  To capture all
documented sites, import a longer upstream window extending to ~3 kb before
the FLO11 ATG.